In [1]:
import pandas as pd
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [12]:
ds = pd.read_csv("../results/merge_res_full.csv", 
                 on_bad_lines='skip',
                 engine='python',
                 quoting=3)

In [ ]:
# More robust parsing with error handling
try:
    ds = pd.read_csv("../results/merge_res_full.csv", 
                     on_bad_lines='skip',
                     engine='python',
                     quoting=3,
                     escapechar='\\')
except:
    # If all else fails, read line by line and skip problematic rows
    import csv
    rows = []
    with open("../results/merge_res_full.csv", 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)
        for i, row in enumerate(reader):
            try:
                if len(row) == len(header):  # Ensure row has correct number of columns
                    rows.append(row)
            except:
                print(f"Skipping problematic row {i+2}")
                continue
    
    ds = pd.DataFrame(rows, columns=header)

In [13]:
import pickle
with open("../results/merge_res_full.pkl", 'wb') as f:
    pickle.dump(ds, f)

In [14]:
import time
start = time.time()
with open("../results/merge_res_full.pkl", 'rb') as f:
    ds2 = pickle.load(f)
print(f"elapsed: {time.time() - start}")

elapsed: 0.5019557476043701


In [15]:
len(ds)

612495

In [ ]:
ds.head()

,commit,created,deleted,poisoned
0,swh:1:rev:734b555b01ad0674ccd10144f327fc6164e7d488,NaN,".idea/12306.iml,.idea/misc.xml,.idea/workspace.xml,.idea/markdown-navigator.xml,.idea/vcs.xml,.idea/modules.xml,.idea/markdown-navigator/profiles_settings.xml,.idea/inspectionProfiles/Project_Default.xml,.idea/markdown-exported-files.xml",False
1,swh:1:rev:34789a5a3ebfe87ed650d6997b8a97914fd28f8c,NaN,tkcode,False
2,swh:1:rev:0d561034eb5c8b52495b75823a7c590c3133bfd3,NaN,tkcode,False
3,swh:1:rev:37f20eda21b68dea8509107e1d69b66bf855564d,NaN,tkcode,False
4,swh:1:rev:99ec619f2cfe0639ca846178b2d6b6eb3f9ea07d,NaN,tkcode,False


In [16]:
ds['poisoned'].value_counts()

poisoned
false                                                                                        611558
true                                                                                              6
Icon                                                                                              2
Chrome/unum-senior-design/.google_apis/speech-v1beta1r40lv1.21.0/speech/libs-sources/Icon         1
Chrome/.metadata/.plugins/org.eclipse.ui.cheatsheets/Icon                                         1
                                                                                              ...  
help/PLASMAETCHINGOFEDGES_files/Icon                                                              1
help/RefelectionTests_files/Icon                                                                  1
help/SheetResistivityMeasurement_files/Icon                                                       1
Project2/testing/MediaReportSat Apr 24 22:44:36 2021                                       

In [17]:
ds_not_poisoned = ds[ds['poisoned']==False][['commit', 'created', 'deleted']]

In [18]:
ds_not_poisoned_created_only = ds_not_poisoned[ds_not_poisoned['deleted'].isna()][['commit', 'created']]
print(f'created only cases: {len(ds_not_poisoned_created_only):,}')

created only cases: 0


In [19]:
ds_not_poisoned_created_only

,commit,created


In [ ]:
all_filenames = []
for created_files in ds_not_poisoned_created_only['created']:
    if pd.notna(created_files):
        file_paths = [path.strip() for path in created_files.split(',')]
        filenames = [os.path.basename(path) for path in file_paths if path.strip()]
        all_filenames.extend(filenames)

In [21]:
filename_counts = pd.Series(all_filenames).value_counts()
print(f'Total unique filenames: {len(filename_counts):,}')
print(f'Total filename occurrences: {len(all_filenames):,}')

Total unique filenames: 3,462
Total filename occurrences: 4,540


In [23]:
filename_counts.head(10)

__init__.py          188
spec.png              50
original.wav          50
finetuned.wav         50
no_finetuning.wav     50
intro.rst             20
datasets.rst          20
README.md             20
configs.rst           20
checkpoint.rst        16
Name: count, dtype: int64

In [35]:
filename_counts[filename_counts.index.str.contains('__init__.py', na=False)]

__init__.py                    188
__init__.py.orig                 8
__init__.py~upstream_master      2
__init__.py~HEAD                 2
__init__.pyi                     1
Name: count, dtype: int64